# Initial Steps: load packages, files and functions

In [1]:
import json, re, time, itertools
from copy import deepcopy
from pathlib import Path
from datetime import datetime
import os
import pandas as pd
import requests

In [2]:
cwd=os.getcwd()
cwd_Raw_Data_outputs=os.path.join(cwd,'RawData')#heres where we store freezes of the raw data
# cwd_Figures=os.path.join(cwd,'Figures')#figures and code for generating them can go here
cwd_Output=os.path.join(cwd,'Output Dataframes')

In [5]:
def GetAwardAmount(input_String, Lists):
    #function takes three arguments; the Reporter output, the destination where we store results, and an additional list for storing a freeze of the data
    GetResults = input_String.find("\"results\"") #find the part of the output detailing grant award amount, found after the "results" block of the ouput
    ResultsList = input_String[GetResults:].replace("},", "")# each grant's information is separated by curly brackets; splitting along curly brackets divides info from each grant
    ResultsList = (ResultsList.split("{\""))[1:]    #saving the individual grant amount as a an element in a list of grants
    for iGrant in ResultsList:# for each grant returned by the query
        Award_Start = iGrant.find("\"award_amount\":")#find the part detailing award amount
        Award_End = iGrant.find("\"project_start_date\":")#find the part that comes after the award amount
        DirectCost = iGrant.find("\"direct_cost_amt\":")
        direct_End = iGrant.find("\"indirect_cost_amt\":")
        Award_string = iGrant[Award_Start:Award_End].replace(",", "").split(":")[1]# the amount of money for grant will be between the part addressed as award amount and the direct cost amount
        directCost = iGrant[DirectCost:direct_End].replace(",", "").split(':', 1)[1]
        indirectCost = iGrant[direct_End:].replace(",", "").split(':', 1)[1]
        if not Award_string == "null": # for some reason, some grants do not have an award amount stored in NIH Reporter
            Lists[0] = Lists[0] + int(Award_string)
            if not directCost == "null":
                Lists[1]=Lists[1]+int(directCost)
            if not "null" in indirectCost:
                indirectCost=indirectCost.replace("}]}","")
                Lists[2]=Lists[2]+int(indirectCost)
    return Lists

In [ ]:

xlsx_path = Path("ShortMeeting history.xlsx")  # <-- change if needed

xls = pd.ExcelFile(xlsx_path)
participants_df = pd.read_excel(xlsx_path, sheet_name=xls.sheet_names[0])  # Participants (first sheet)
meetings_df      = pd.read_excel(xlsx_path, sheet_name="Meetings")

participants_df.iloc[ 1:2] = "Targeting Lipid Biology in Cancer"
participants_df.tail(5)

,Participant,Meeting,Type,First Name,Last Name,Suffix,Institution,Title
54,"Elsa Flores, PhD",2005 Scholar Retreat,Scholar,Elsa,Flores,PhD,MD Anderson Cancer Center,NaN
55,"Kimryn Rathmell, MD, PhD",2005 Scholar Retreat,Scholar,Kimryn,Rathmell,"MD, PhD",Vanderbilt University Medical Center,NaN
56,"Masashi Narita, MD, PhD",2005 Scholar Retreat,Scholar,Masashi,Narita,"MD, PhD",Cambridge Institute,NaN
57,"Jan Karlseder, PhD",2005 Scholar Retreat,Scholar,Jan,Karlseder,PhD,Salk Institute,NaN
58,"James Amatruda, MD, PhD",2005 Scholar Retreat,Scholar,James,Amatruda,"MD, PhD",Memorial Sloan Kettering Cancer Center,NaN


In [ ]:
_TITLE_RE = re.compile(r"^(dr\.?|prof\.?|mr\.?|ms\.?|mrs\.?)\s+", re.I)
# Strips trailing comma-separated credentials (extend list if you have others)
_CRED_RE = re.compile(r"(?:,?\s*(?:MD|M\.D\.|PhD|Ph\.D\.|DO|D\.O\.|MPH|MS|MSc|MBA|JD|DDS|DVM|RN))+$", re.I)

def normalize_meeting_title(x: str) -> str:
    if pd.isna(x): return None
    s = str(x).strip()
    if len(s) >= 2 and s[0] == s[-1] and s[0] in {"'", '"'}:
        s = s[1:-1].strip()
    return re.sub(r"\s+", " ", s)

def strip_titles_and_credentials(name: str) -> str:
    if pd.isna(name): return None
    s = str(name).strip()
    s = _TITLE_RE.sub("", s)      # leading "Dr.", "Prof.", etc.
    s = _CRED_RE.sub("", s)       # trailing ", MD, PhD" etc.
    return re.sub(r"\s+", " ", s).strip()

def split_chair_names(chairs_cell) -> list[str]:
    if pd.isna(chairs_cell): return []
    s = re.sub(r"\s+", " ", str(chairs_cell).strip())
    parts = [p.strip() for p in s.split(";") if p.strip()]
    chairs = []
    for p in parts:
        for item in re.split(r"\s+(?:and|&)\s+", p):
            item = item.strip()
            if not item: 
                continue
            item = re.sub(r"\s+of\s+.+$", "", item).strip()   # drop trailing institution
            item = strip_titles_and_credentials(item)         # drop titles/credentials
            if item:
                chairs.append(item)
    out, seen = [], set()
    for c in chairs:
        if c not in seen:
            seen.add(c); out.append(c)
    return out

In [ ]:
meetings_df = meetings_df.copy()
participants_df = participants_df.copy()

# Normalize meeting names
meetings_df["MeetingTopic_norm"] = meetings_df["Meeting Topic"].map(normalize_meeting_title)
participants_df["Meeting_norm"]  = participants_df["Meeting"].map(normalize_meeting_title)

# Strip titles/credentials from participant names (THIS is the key change)
participants_df["Participant_clean"] = participants_df["Participant"].map(strip_titles_and_credentials)

# Map normalized meeting topic -> year (first non-null year per meeting)
meeting_to_year = (
    meetings_df.dropna(subset=["MeetingTopic_norm", "Year"])
               .drop_duplicates(subset=["MeetingTopic_norm"])
               .set_index("MeetingTopic_norm")["Year"]
               .to_dict()
)

# Attach year to participants
participants_df["Year"] = participants_df["Meeting_norm"].map(meeting_to_year)

# Build dict: "Meeting Topic (Year)" -> sorted unique participant_clean names
def make_meeting_year_key(meeting_norm, year):
    if meeting_norm is None:
        return None
    if pd.isna(year):
        return f"{meeting_norm} (Year Unknown)"
    y = int(year) if float(year).is_integer() else year
    return f"{meeting_norm} ({y})"

tmp = participants_df.dropna(subset=["Meeting_norm", "Participant_clean"]).copy()
tmp["MeetingYearKey"] = [make_meeting_year_key(m, y) for m, y in zip(tmp["Meeting_norm"], tmp["Year"])]

meeting_attendees_dict = (
    tmp.groupby("MeetingYearKey")["Participant_clean"]
       .apply(lambda s: sorted(set(s.dropna().astype(str).str.strip())))
       .to_dict()
)

# optional: ensure chairs are included too (also title/credential stripped)
meetings_df["Chairs_clean"] = meetings_df["Meeting Chairs"].apply(split_chair_names)
for _, row in meetings_df.iterrows():
    topic = row.get("MeetingTopic_norm")
    year = row.get("Year")
    if not topic:
        continue
    key = make_meeting_year_key(topic, year)
    if key not in meeting_attendees_dict:
        meeting_attendees_dict[key] = []
    for chair in (row.get("Chairs_clean") or []):
        meeting_attendees_dict[key].append(chair)
    meeting_attendees_dict[key] = sorted(set(meeting_attendees_dict[key]))

# Preview
list(meeting_attendees_dict.items())[:5]

,Participant,Meeting,Year
0,"Alison Ringel, PhD",Targeting Lipid Biology in Cancer,2023
1,Targeting Lipid Biology in Cancer,Targeting Lipid Biology in Cancer,2023
2,"Bart Vanhaesebroeck, PhD",Targeting Lipid Biology in Cancer,2023
3,"Christina Mitchell, MB BS, PhD",Targeting Lipid Biology in Cancer,2023
4,"Neil Vasan, MD, PhD",Targeting Lipid Biology in Cancer,2023
5,"Prof Banafshe Larijani , PhD",Targeting Lipid Biology in Cancer,2023
6,"Ray Blind,",Targeting Lipid Biology in Cancer,2023
7,"Brooke Emerling, PhD",Targeting Lipid Biology in Cancer,2023
8,"Gretchen Alicea, PhD",Targeting Lipid Biology in Cancer,2023
9,"Sarah Skuli,",Targeting Lipid Biology in Cancer,2023


In [37]:
# Cell 4 — (1) create dict keyed by "Meeting Topic (Year)" with list of participant full names

# Keep only rows that have a meeting + participant name
p = participants_df.dropna(subset=["Meeting_norm", "Participant"]).copy()

# Build a key string like "Targeting Lipid Biology in Cancer (2021)"
def make_meeting_year_key(meeting_norm, year):
    y = "" if pd.isna(year) else str(int(year)) if float(year).is_integer() else str(year)
    return f"{meeting_norm} ({y})" if y else f"{meeting_norm} (Year Unknown)"

p["MeetingYearKey"] = [make_meeting_year_key(m, y) for m, y in zip(p["Meeting_norm"], p["Year"])]

meeting_attendees_dict = (
    p.groupby("MeetingYearKey")["Participant"]
     .apply(lambda s: sorted(set(s.dropna().astype(str).str.strip())))
     .to_dict()
)

# Example: show first 5 keys
list(meeting_attendees_dict["Targeting Lipid Biology in Cancer (2023)"])

['Alison Ringel, PhD',
 'Bart Vanhaesebroeck, PhD',
 'Brooke Emerling, PhD',
 'Christina Mitchell, MB BS, PhD',
 'David Fruman, PhD',
 'Emilio Hirsch, PhD',
 'Gretchen Alicea, PhD',
 'Hua Eleanor Yu, PhD',
 'Jeremy Baskin, PhD',
 'Karen Dixon,',
 'Livia  Schiavinato Eberlin, PhD',
 'Neil Vasan, MD, PhD',
 'Prof Banafshe  Larijani , PhD',
 'Ray Blind,',
 'Sarah  Skuli,',
 'Tamas Balla, MD, PhD',
 'Targeting Lipid Biology in Cancer',
 'Vytas Bankaitis, PhD']

In [33]:
REPORTER_SEARCH_URL = "https://api.reporter.nih.gov/v2/projects/search"
_SUFFIXES = {"jr","sr","ii","iii","iv","md","phd","mph","ms","m.d.","ph.d.","dr"}

def norm(s): 
    return re.sub(r"\s+", " ", str(s or "").strip())

def strip_cred(name):
    s = norm(name)
    s = re.sub(r"\([^)]*\)", "", s)                # remove ( ... )
    s = re.sub(r"^(dr\.?|prof\.?)\s+", "", s, flags=re.I)
    parts = [p.strip() for p in s.split(",") if p.strip()]
    # keep "Last, First" if present; otherwise keep first chunk before credentials
    if len(parts) >= 2: 
        return f"{parts[0]}, {parts[1].split()[0]}"
    return parts[0] if parts else s

def split_first_last(name):
    """Return (first, last) best-effort from many formats."""
    s = strip_cred(name)
    if not s: return "", ""
    if "," in s:                                    # "Last, First"
        last, first = [x.strip() for x in s.split(",", 1)]
        first = first.split()[0] if first else ""
        return first.lower(), last.lower()

    toks = [t for t in s.replace(".", " ").split() if t]
    toks = [t for t in toks if t.lower() not in _SUFFIXES]
    if len(toks) == 1: return "", toks[0].lower()
    return toks[0].lower(), toks[-1].lower()

def build_attendee_index(attendee_names):
    """
    Build dict keyed by last name -> list of candidate attendee records.
    Each record has: display, first, last, first_initial
    """
    idx = {}
    for disp in attendee_names:
        first, last = split_first_last(disp)
        if not last: 
            continue
        rec = {"display": disp, "first": first, "last": last, "fi": (first[0] if first else "")}
        idx.setdefault(last, []).append(rec)
    return idx

def extract_pi_name_records(principal_investigators):
    """
    Normalize PI roster into list of dicts: {"first":..., "last":..., "raw":...}
    Handles list of dicts or strings (best-effort).
    """
    out = []
    if not isinstance(principal_investigators, list):
        return out
    for pi in principal_investigators:
        if isinstance(pi, dict):
            fn = norm(pi.get("first_name") or pi.get("firstName") or "")
            ln = norm(pi.get("last_name")  or pi.get("lastName")  or "")
            raw = pi.get("pi_name") or pi.get("name") or f"{fn} {ln}".strip()
            if not ln and raw:
                f2, l2 = split_first_last(raw)
                fn, ln = f2, l2
            out.append({"first": fn.lower(), "last": ln.lower(), "raw": raw})
        else:
            f2, l2 = split_first_last(pi)
            out.append({"first": f2, "last": l2, "raw": str(pi)})
    return out

def match_attendees_in_pi_roster(attendee_index, pi_records):
    """
    Returns a set of attendee display names that match ANY PI in roster.
    Matching rule:
      - last name must match exactly
      - if attendee has a first name:
          * match if PI first startswith attendee first OR attendee first startswith PI first
          * OR first initial matches
      - if attendee first unknown: last-only match (looser)
    """
    matched = set()
    for pi in pi_records:
        last = pi.get("last", "")
        if not last or last not in attendee_index:
            continue

        pi_first = pi.get("first", "")
        pi_fi = pi_first[0] if pi_first else ""

        for a in attendee_index[last]:
            if a["first"]:
                # strong-ish match: first initial OR prefix match
                if (a["fi"] and pi_fi and a["fi"] == pi_fi) or \
                   (pi_first and a["first"] and (pi_first.startswith(a["first"]) or a["first"].startswith(pi_first))):
                    matched.add(a["display"])
            else:
                # last-only attendee (rare) -> accept
                matched.add(a["display"])
    return matched

# -----------------------
# RePORTER paging
# -----------------------

def reporter_search_all_pages(payload, sleep=0.25, limit=500, verbose=False):
    params = deepcopy(payload); params.update({"offset": 0, "limit": limit})
    pages, total = [], None
    while True:
        r = requests.post(REPORTER_SEARCH_URL, json=params, timeout=60)
        r.raise_for_status()
        page = r.json(); pages.append(page)
        total = total or page.get("meta", {}).get("total", 0)
        off = page.get("meta", {}).get("offset", params["offset"])
        cnt = page.get("meta", {}).get("count", len(page.get("results", [])))
        if verbose: print(f"      page offset={off} count={cnt} total={total}")
        if cnt == 0 or off + cnt >= total: break
        params["offset"] = off + cnt; time.sleep(sleep)
    return {"total": int(total or 0), "pages": pages}

def meeting_year_from_key(meeting_key):
    m = re.search(r"\((\d{4})\)\s*$", str(meeting_key))
    return int(m.group(1)) if m else None

def five_year_bins(center_year, n_before=1, n_after=3):
    return ([(center_year-5*i, center_year-5*(i-1)) for i in range(n_before, 0, -1)] +
            [(center_year+5*i, center_year+5*(i+1)) for i in range(0, n_after)])

def fiscal_years_for_bin(start, end):
    return list(range(int(start), int(end) + 1))

def pick_cost(proj):
    c = proj.get("fy_total_cost")
    if c is None: c = proj.get("award_amount")
    return int(c or 0)

# -----------------------
# MAIN: multi-PI awards with >=2 meeting attendees on PI roster
# -----------------------

def multi_pi_awards_2plus_attendees(
    meeting_attendees_dict,
    NIH_param_template,
    n_bins_before=1,
    n_bins_after=3,
    sleep=0.25,
    verbose=True,
    verbose_show_first_match_per_bin=True
):
    summary_rows, detail_rows = [], []
    print("\n=== START: Multi-PI awards containing >=2 meeting attendees as PIs ===")
    print("Method: query each attendee -> examine PI roster -> keep awards where >=2 attendees match.\n")

    for meeting_key, attendees in meeting_attendees_dict.items():
        year = meeting_year_from_key(meeting_key)
        if year is None:
            print(f"\nMeeting: {meeting_key}\n  ⚠ No year found; skipping.")
            continue

        attendee_index = build_attendee_index(attendees)
        attendee_count = sum(len(v) for v in attendee_index.values())
        if verbose:
            print("\n===================================================")
            print(f"Meeting: {meeting_key}")
            print(f"Meeting year: {year}")
            print(f"Attendees parsable (indexed): {attendee_count}")

        for start, end in five_year_bins(year, n_bins_before, n_bins_after):
            bin_label = f"{start}-{end}"
            fys = fiscal_years_for_bin(start, end)
            if verbose:
                print(f"\n--- Bin {bin_label} (FY {fys[0]}..{fys[-1]}) ---")

            seen_appl_ids = set()      # dedupe per bin
            kept_appl_ids = set()
            appl_to_cost = {}
            bin_queries = 0
            showed_example = False

            # Query each attendee (last name + optional first name)
            for last, recs in attendee_index.items():
                for rec in recs:
                    bin_queries += 1
                    payload = deepcopy(NIH_param_template)
                    payload.setdefault("criteria", {})
                    c = payload["criteria"]
                    c["multi_pi_only"] = True
                    c["fiscal_years"] = fys
                    c["pi_names"] = [{"any_name": rec["last"], "first_name": rec["first"] or ""}]
                    c.pop("advanced_text_search", None)

                    payload["include_fields"] = [
                        "appl_id","fiscal_year","project_num","project_title",
                        "fy_total_cost","award_amount","principal_investigators"
                    ]

                    res = reporter_search_all_pages(payload, sleep=sleep, verbose=False)
                    if res["total"] == 0: 
                        continue
                    print(res)
                    # Validate: does PI roster contain >=2 meeting attendees?
                    for page in res["pages"]:
                        for proj in page.get("results", []):
                            appl = proj.get("appl_id")
                            if appl is None: 
                                continue
                            if appl in seen_appl_ids:
                                # already processed for this bin; skip heavy matching again
                                continue

                            pi_records = extract_pi_name_records(proj.get("principal_investigators"))
                            matched_attendees = match_attendees_in_pi_roster(attendee_index, pi_records)

                            if len(matched_attendees) >= 2:
                                kept_appl_ids.add(appl)
                                cost = pick_cost(proj)
                                appl_to_cost[appl] = max(appl_to_cost.get(appl, 0), cost)

                                detail_rows.append({
                                    "MeetingYearKey": meeting_key,
                                    "Bin": bin_label,
                                    "ApplId": appl,
                                    "FiscalYear": proj.get("fiscal_year"),
                                    "ProjectNum": proj.get("project_num"),
                                    "ProjectTitle": proj.get("project_title"),
                                    "AwardAmount": cost,
                                    "MatchedAttendees": sorted(matched_attendees),
                                    "PrincipalInvestigators_raw": proj.get("principal_investigators")
                                })

                                if verbose and verbose_show_first_match_per_bin and not showed_example:
                                    showed_example = True
                                    print("  ✅ Example kept award:")
                                    print(f"     appl_id={appl} | matched_attendees={sorted(matched_attendees)[:6]}{'...' if len(matched_attendees)>6 else ''}")

                            # mark this appl_id as examined (so we don’t re-check it via other attendee queries)
                            seen_appl_ids.add(appl)

                    time.sleep(sleep)

            unique_awards = len(kept_appl_ids)
            total_award_amount = sum(appl_to_cost.get(appl, 0) for appl in kept_appl_ids)

            if verbose:
                print(f"Bin done: attendee-queries={bin_queries}")
                print(f"  Unique awards with >=2 attendee PIs: {unique_awards}")
                print(f"  Total award amount (dedup by appl_id): ${total_award_amount:,}")

            summary_rows.append({
                "MeetingYearKey": meeting_key,
                "MeetingYear": year,
                "Bin": bin_label,
                "Attendees_indexed": attendee_count,
                "Attendee_queries": bin_queries,
                "Unique_awards_2plus_attendees": unique_awards,
                "Total_award_amount": total_award_amount
            })

    print("\n=== COMPLETE ===")
    return pd.DataFrame(summary_rows), pd.DataFrame(detail_rows)

In [34]:
NIH_param = {"criteria": {}}  # minimal template is fine

summary_df, details_df = multi_pi_awards_2plus_attendees(
    meeting_attendees_dict,
    NIH_param_template=NIH_param,
    n_bins_before=1,
    n_bins_after=3,
    sleep=0.25,
    verbose=True
)
summary_df.sort_values(["MeetingYearKey","Bin"]).head(20)


=== START: Multi-PI awards containing >=2 meeting attendees as PIs ===
Method: query each attendee -> examine PI roster -> keep awards where >=2 attendees match.


Meeting: 2005 Scholar Retreat (2005)
Meeting year: 2005
Attendees parsable (indexed): 17

--- Bin 2000-2005 (FY 2000..2005) ---
Bin done: attendee-queries=17
  Unique awards with >=2 attendee PIs: 0
  Total award amount (dedup by appl_id): $0

--- Bin 2005-2010 (FY 2005..2010) ---
{'total': 2, 'pages': [{'meta': {'search_id': 'om41GthEdEyx_QL4fBEIBg', 'total': 2, 'offset': 0, 'limit': 500, 'sort_field': None, 'sort_order': 'ASC', 'sorted_by_relevance': True, 'properties': {'URL': 'https:/reporter.nih.gov/search/om41GthEdEyx_QL4fBEIBg/projects'}}, 'results': [{}, {}]}]}
{'total': 2, 'pages': [{'meta': {'search_id': 'P2qcwaHjQU-XuGkLS6pLKQ', 'total': 2, 'offset': 0, 'limit': 500, 'sort_field': None, 'sort_order': 'ASC', 'sorted_by_relevance': True, 'properties': {'URL': 'https:/reporter.nih.gov/search/P2qcwaHjQU-XuGkLS6pLKQ/p

,MeetingYearKey,MeetingYear,Bin,Attendees_indexed,Attendee_queries,Unique_awards_2plus_attendees,Total_award_amount
0,2005 Scholar Retreat (2005),2005,2000-2005,17,17,0,0
1,2005 Scholar Retreat (2005),2005,2005-2010,17,17,0,0
2,2005 Scholar Retreat (2005),2005,2010-2015,17,17,0,0
3,2005 Scholar Retreat (2005),2005,2015-2020,17,17,0,0
4,3D Chromosomal Architecture and Nuclear Topolo...,2019,2014-2019,24,24,0,0
5,3D Chromosomal Architecture and Nuclear Topolo...,2019,2019-2024,24,24,0,0
6,3D Chromosomal Architecture and Nuclear Topolo...,2019,2024-2029,24,24,0,0
7,3D Chromosomal Architecture and Nuclear Topolo...,2019,2029-2034,24,24,0,0
8,Targeting Lipid Biology in Cancer (2023),2023,2018-2023,18,18,0,0
9,Targeting Lipid Biology in Cancer (2023),2023,2023-2028,18,18,0,0
